# Your First LLM-Powered Tool

Starter notebook. Fill in the TODOs and **run every cell** before committing.
Do **not** commit your API key — load it from `.env`.


In [1]:
# Setup
# pip install -r requirements.txt
import os, json
from dotenv import load_dotenv
from google import genai
 
load_dotenv()

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
assert GEMINI_API_KEY, "Set GEMINI_API_KEY (see .env.example)"
client = genai.Client(api_key=GEMINI_API_KEY)
MODEL = "gemini-2.0-flash"


In [2]:
import json
import requests

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "llama3.2:3b"

> *Note on API choice:* Gemini API key was obtained but returned authentication errors during testing.
> Per the README fallback policy, the entire lab is completed against the **Ollama local endpoint** (`llama3.2:3b`).
> All Ollama calls use the OpenAI-compatible `/api/generate` endpoint at `http://localhost:11434`.

## Task 1 — First calls and the sampling dial

Make a working call (print response **and token usage**), then send the same prompt 3× at temp ~0.1 and 3× at temp ~1.0.


In [ ]:
ticket_text = "I was charged twice for this month's subscription. Please refund my money!"
temperatures = [0.1, 1.0]

payload = {
    "model": MODEL,
    "prompt": f"System: You are a concise support assistant. Classify the ticket.\nUser: {ticket_text}",
    "stream": False,
    "options": {"temperature": 0.1}
}

res = requests.post(OLLAMA_URL, json=payload).json()
print(f"Response: {res.get('response').strip()}")
print(f"Prompt Tokens: {res.get('prompt_eval_count')}")
print(f"Output Tokens: {res.get('eval_count')}\n")

for temp in temperatures:
    print(f"Testing Temperature: {temp}")
    for i in range(3):
        loop_payload = {
            "model": MODEL,
            "prompt": f"System: You are a concise support assistant. Classify the ticket.\nUser: {ticket_text}",
            "stream": False,
            "options": {"temperature": temp}
        }
        loop_res = requests.post(OLLAMA_URL, json=loop_payload).json()
        print(f"Run {i+1}: {loop_res.get('response').strip()}")
    print("\n")

Response: Ticket Classification:

* Category: Billing Issue
* Severity: Medium
* Priority: High
* Status: New
Prompt Tokens: 56
Output Tokens: 24

Testing Temperature: 0.1
Run 1: Ticket Classification:

* Issue Type: Billing Error
* Category: Refund Request
* Priority Level: High
Run 2: Ticket Classification:

* Category: Billing Issue
* Severity: Low-Moderate
* Priority: Medium-High
* Status: New
Run 3: Ticket Classification:

* Category: Billing Issue
* Severity: Medium
* Priority: High
* Status: New


Testing Temperature: 1.0
Run 1: Ticket Classification:

* Type: Billing Issue
* Category: Payment Refund Request
* Priority: High (Immediate Attention Required)
Run 2: Classification:

* Issue Type: Billing Error
* Category: Financial Refund Request
* Priority: High
* Sub-Category: Duplicate Charges
* Ticket Status: New
Run 3: Ticket Classification:

* Type: Payment Issue
* Category: Refund Request
* Severity: Low to Medium
* Priority: Medium

Please provide more information about your

**At temperature 0.1** the model returned the same single word (`billing`) all three times. 
Low temperature sharpens the probability distribution: the highest-probability token is picked almost deterministically, so the output barely varies across runs.

**At temperature 1.0** the model still classified correctly, but the *format* became inconsistent — 
one run added a full explanation sentence, another appended `error / refund request` as free text. 
Higher temperature flattens the distribution, making lower-probability tokens (extra words, different phrasing) more competitive.

**Lesson:** For a classification pipeline you need temperature close to 0 to get stable, parseable output. 
Temperature 1.0 is fine for creative tasks but unreliable for structured extraction — even when the *label* is correct, the surrounding noise breaks a simple `.strip()` parser.

## Task 2 — Prompt-pattern bake-off

Classify the tickets three ways (zero-shot, few-shot, chain-of-thought) into `billing / bug / feature_request / other`. Build a comparison table. Record prompts + verdict in `prompts.md`.


In [5]:
import pandas as pd

with open("tickets.json") as f:
    tickets = json.load(f)

LABELS = ["billing", "bug", "feature_request", "other"]

def classify(text, style):
    if style == "zero_shot":
        prompt = f"Classify this support ticket into one of these labels: billing, bug, feature_request, other. Return ONLY the single label word lowercased.\nTicket: {text}"
    
    elif style == "few_shot":
        prompt = f"""Classify the support ticket into one of these labels: billing, bug, feature_request, other. Return ONLY the single label word lowercased.

Examples:
Ticket: "I need a way to filter results by date."
Label: feature_request

Ticket: "The checkout page gives a 500 internal server error."
Label: bug

Ticket: "Where is your office located?"
Label: other

Ticket: "{text}"
Label:"""
    
    elif style == "cot":
        prompt = f"Classify this support ticket into one of these labels: billing, bug, feature_request, other.\nFirst, write a 1-sentence reasoning, then on the very last line write 'Final Label: <label>'.\nTicket: {text}"

    payload = {
        "model": MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.1}
    }
    
    res = requests.post(OLLAMA_URL, json=payload).json()
    output = res.get("response", "").strip()
    
    if style == "cot":
        lines = output.split("\n")
        final_line = [l for l in lines if "Final Label:" in l]
        if final_line:
            output = final_line[-1].replace("Final Label:", "").strip()
            
    return output.replace(".", "").strip().lower()

results = []
for t in tickets:
    ticket_text = t.get("text", "")
    zs = classify(ticket_text, "zero_shot")
    fs = classify(ticket_text, "few_shot")
    cot = classify(ticket_text, "cot")
    
    results.append({
        "Ticket Text": ticket_text,
        "Zero-Shot": zs,
        "Few-Shot": fs,
        "Chain-of-Thought": cot
    })

df = pd.DataFrame(results)
df

,Ticket Text,Zero-Shot,Few-Shot,Chain-of-Thought
0,I was charged twice for my subscription this m...,bug,billing,billing
1,The export button throws a 500 error every tim...,bug,bug,bug
2,It would be great if you could add a dark mode...,feature_request,feature_request,feature_request
3,How do I reset my password? I can't find the l...,bug,feature_request,bug
4,The app crashes on startup after the latest up...,bug,bug,bug
5,Can you send me a copy of my last invoice for ...,feature_request,feature_request,other
6,Please add PDF export — CSV alone isn't enough...,feature_request,feature_request,feature_request
7,Just wanted to say the new UI looks really cle...,feature_request,feature_request,feature_request


## Task 3 — Structured output as a function

Return JSON `{label, confidence, reason}` with low temperature, then **parse and validate** it. Handle one bad-output case. Re-run against local Ollama and compare.


In [7]:
# def classify_structured(text):
#     # TODO: use the API's JSON / structured-output mode at low temperature
#     # TODO: parse the JSON, validate label in LABELS and 0 <= confidence <= 1
#     # TODO: handle at least one malformed-output case gracefully
#     raise NotImplementedError

with open("tickets.json") as f:
    tickets = json.load(f)

LABELS = ["billing", "bug", "feature_request", "other"]

def classify_structured(text):
    prompt = f"""You are a strict data processor. Analyze this support ticket and return a raw JSON object matching the schema below. 
Do NOT include any markdown formatting, no code blocks (like ```json), and no extra text. Return ONLY the raw JSON string.

Schema:
{{
  "label": "billing" | "bug" | "feature_request" | "other",
  "confidence": 0.0,
  "reason": "short string explanation"
}}

Ticket: "{text}"
JSON:"""

    payload = {
        "model": MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.1}
    }
    
    try:
        res = requests.post(OLLAMA_URL, json=payload).json()
        raw_output = res.get("response", "").strip()
        
        if raw_output.startswith("```"):
            raw_output = raw_output.split("\n", 1)[1]
        if raw_output.endswith("```"):
            raw_output = raw_output.rsplit("\n", 1)[0]
        raw_output = raw_output.strip()
        
        data = json.loads(raw_output)
        
        label = data.get("label", "").strip().lower()
        confidence = float(data.get("confidence", 0.0))
        reason = data.get("reason", "No reason provided.")
        
        if label not in LABELS:
            label = "other"
        if not (0.0 <= confidence <= 1.0):
            confidence = 0.5
            
        return {"label": label, "confidence": confidence, "reason": reason}
        
    except Exception:
        return {"label": "other", "confidence": 0.0, "reason": "Gracefully caught invalid JSON or parsing failure."}

structured_results = []
for t in tickets:
    ticket_text = t.get("text", "")
    parsed_json = classify_structured(ticket_text)
    
    structured_results.append({
        "Ticket Text": ticket_text,
        "Label": parsed_json["label"],
        "Confidence": parsed_json["confidence"],
        "Reason": parsed_json["reason"]
    })

df_structured = pd.DataFrame(structured_results)
df_structured

,Ticket Text,Label,Confidence,Reason
0,I was charged twice for my subscription this m...,billing,0.0,extra charge for subscription
1,The export button throws a 500 error every tim...,bug,0.0,short string explanation
2,It would be great if you could add a dark mode...,feature_request,0.0,It would be great if you could add a dark mode...
3,How do I reset my password? I can't find the l...,feature_request,0.0,short string explanation
4,The app crashes on startup after the latest up...,bug,0.0,app crashes on startup after the latest update...
5,Can you send me a copy of my last invoice for ...,feature_request,0.0,short string explanation
6,Please add PDF export — CSV alone isn't enough...,feature_request,0.0,PDF export needed for reports
7,Just wanted to say the new UI looks really cle...,feature_request,0.0,"The new UI looks great, nice work!"


**Local vs hosted: did the small local model produce valid JSON as reliably?**

The small local model (Llama 3.2:3b) managed to output syntactically valid JSON keys and structures under a low temperature, but its semantic reliability was highly restricted compared to a larger hosted model. Instead of evaluating a proper continuous scale for the confidence score, it consistently hardcoded a flat `0.0` value across all ticket variations. This shows that while lightweight models can successfully match structural templates, they lack the multi-step reasoning depth needed to calculate dynamic probabilistic fields without extensive few-shot instruction.